# 06 · App de Notas de Campo — Databricks Apps + Lakebase

**Objetivo:** desplegar la app `cfe-notas-obra` (codigo en `apps/cfe-notas-obra` del repo) para que los residentes registren notas de campo sobre las obras, con Lakebase Postgres como base transaccional.

En este laboratorio despliegas la app de **Notas de Campo CFE** usando la interfaz de Databricks. Solo hay **una celda de codigo** (paso 4), para copiar el catalogo de obras de la capa Gold a Lakebase.

```
Usuario --> App (app_v3.py) --> Lakebase Postgres (esquema public)
                                  |- obras_catalogo  (lo cargas en el paso 4)
                                  |- notas_obra      (lo crea la app)
                                  '- bitacora_app    (lo crea la app)
                                          |
            Job: sync_lakebase_to_delta   v   (paso 8)
            Tablas Delta *_sync en Unity Catalog para analitica
```

| Paso | Donde | Que haces |
| --- | --- | --- |
| 1 | Lakebase | Crear el proyecto de Postgres |
| 2 | Apps | Crear la app (sin desplegar) |
| 3 | Apps | Ligar Lakebase como recurso de la app |
| 4 | Este notebook | Cargar el catalogo de obras (unica celda de codigo) |
| 5 | Lakebase SQL Editor | Dar permisos a la app |
| 6 | Editor de archivos | Completar `app.yaml` |
| 7 | Apps | Desplegar y probar la app |
| 8 | Jobs & Pipelines | Programar la sincronizacion a Delta |

**Antes de empezar**
* Debe existir `gold_fact_obras` (pipeline del lab 03).
* Usa un nombre propio para tus recursos, por ejemplo `cfe-notas-<tus-iniciales>`, para no chocar con otros participantes.
* Ten a la mano un bloc de notas: vas a copiar **3 valores** (endpoint, host y client ID de la app).

> Los nombres de botones y menus pueden variar ligeramente segun la version de la interfaz.

## Paso 1 - Crear el proyecto de Lakebase

1. En el selector de aplicaciones (arriba a la derecha) o en el menu lateral, entra a **Lakebase Postgres**.
2. Clic en **Create project** (Crear proyecto).
3. Nombre: `cfe-notas-<tus-iniciales>`. Version de Postgres: la mas reciente. Deja el resto por defecto.
4. Clic en **Create**. En segundos se crean automaticamente:
   * el branch **production**
   * el compute **primary** (lectura/escritura, con autoescalado y scale-to-zero)
   * la base de datos **databricks_postgres**
5. Dentro del proyecto, abre el branch **production** y clic en **Connect** (Conectar). Anota:
   * **Host** -> ejemplo: `ep-xxxx.database.<region>.azuredatabricks.net` -> sera tu `PGHOST`
   * **Endpoint** -> `projects/<proyecto>/branches/production/endpoints/primary` -> sera tu `LAKEBASE_ENDPOINT`

> Si el instructor ya creo un proyecto compartido (por ejemplo `lakebaseworkshop`), puedes usarlo y saltar este paso.

## Paso 2 - Crear la app (sin desplegar)

1. Menu lateral -> **Compute** -> pestana **Apps** -> **Create app**.
2. Elige **Create a custom app** (app personalizada).
3. Nombre: `cfe-notas-<tus-iniciales>` (minusculas y guiones). Descripcion: *Notas de campo CFE con Lakebase*.
4. Clic en **Create app**. Espera a que el compute de la app quede **Running** (2-4 min).
5. En la pagina de la app, pestana **Authorization** (o en los detalles de la app), anota el **Client ID** del service principal de la app -> sera tu `PGUSER`.

> La app corre con su propio **service principal**: una identidad tecnica, distinta de tu usuario, a la que hay que darle permisos.

## Paso 3 - Ligar Lakebase como recurso de la app

1. En la pagina de la app, clic en **Edit** (Editar).
2. En **App resources**, clic en **Add resource** -> **Database** / **Lakebase**.
3. Selecciona tu proyecto, el branch **production** y la base de datos **databricks_postgres**.
4. Permiso: **Can connect and create**.
5. Guarda los cambios (**Save**).

Con esto Databricks crea en Postgres un **rol para el service principal de la app** y le permite conectarse y crear tablas. No necesitas crear usuarios ni contrasenas: la app usa tokens OAuth temporales.

> Puedes confirmarlo en Lakebase: proyecto -> branch **production** -> **Roles**. Debe aparecer un rol con el Client ID del paso 2.

## Paso 4 - Cargar el catalogo de obras (unica parte con codigo)

La app muestra la lista de obras leyendo la tabla `obras_catalogo` en Lakebase. Las dos celdas siguientes copian `IdObra` y `NombreObra` desde `gold_fact_obras` (Delta) hacia Lakebase.

1. Escribe el nombre de tu proyecto de Lakebase en el widget **Proyecto Lakebase** (arriba del notebook).
2. Ejecuta las dos celdas siguientes (**Shift + Enter**).
3. La segunda celda imprime tu `LAKEBASE_ENDPOINT` y `PGHOST`: usalos en el paso 6.

> Alternativa sin codigo: crear un **synced table** desde Catalog Explorer (tabla -> **Create** -> **Synced table**). Para este laboratorio usamos la carga directa porque la app espera la tabla en el esquema `public`.

In [0]:
%pip install "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1.0" -q
dbutils.library.restartPython()

In [0]:
import os, psycopg
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("proyecto_lakebase", "lakebaseworkshop", "Proyecto Lakebase")
PROJECT_ID = dbutils.widgets.get("proyecto_lakebase").strip()
ENDPOINT = f"projects/{PROJECT_ID}/branches/production/endpoints/primary"

# Catalogo y esquema de trabajo desde notebooks/config.py del repo
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_cfg = f"/Workspace{os.path.dirname(_nb)}/config.py"
exec(open(_cfg).read()) if os.path.exists(_cfg) else None
CATALOG, WORK_SCHEMA = globals().get("CATALOG", "dipidev"), globals().get("WORK_SCHEMA", "default")

w = WorkspaceClient()
PGHOST = w.postgres.get_endpoint(name=ENDPOINT).status.hosts.host

obras = [(int(r.IdObra), r.NombreObra) for r in
         spark.table(f"{CATALOG}.{WORK_SCHEMA}.gold_fact_obras")
              .select("IdObra", "NombreObra").dropDuplicates(["IdObra"]).collect()]

cred = w.postgres.generate_database_credential(endpoint=ENDPOINT)
with psycopg.connect(host=PGHOST, dbname="databricks_postgres", user=w.current_user.me().user_name,
                     password=cred.token, sslmode="require") as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS public.obras_catalogo (id_obra INT PRIMARY KEY, nombre_obra TEXT)")
    conn.execute("TRUNCATE public.obras_catalogo")
    conn.cursor().executemany("INSERT INTO public.obras_catalogo VALUES (%s, %s)", obras)
    total = conn.execute("SELECT count(*) FROM public.obras_catalogo").fetchone()[0]

print(f"obras_catalogo cargada: {total} obras (Gold: {len(obras)})")
print("\nValores para app.yaml (paso 6):")
print(f"  LAKEBASE_ENDPOINT = {ENDPOINT}")
print(f"  PGHOST            = {PGHOST}")

## Paso 5 - Dar permisos a la app (Lakebase SQL Editor)

La tabla `obras_catalogo` la creaste tu, asi que la app todavia no puede leerla.

1. Ve a **Lakebase Postgres** -> tu proyecto -> branch **production** -> **SQL Editor**.
2. Pega estas dos lineas, reemplaza `<CLIENT_ID>` por el Client ID de la app (paso 2) **conservando las comillas dobles** y ejecuta:

```sql
GRANT USAGE, CREATE ON SCHEMA public TO "<CLIENT_ID>";
GRANT SELECT ON public.obras_catalogo TO "<CLIENT_ID>";
```

3. Verifica que el catalogo se ve bien:

```sql
SELECT * FROM public.obras_catalogo ORDER BY id_obra LIMIT 10;
```

## Paso 6 - Completar `app.yaml`

`app.yaml` le indica a la app que programa ejecutar y como conectarse a Lakebase.

1. En el explorador de archivos (panel izquierdo), abre la carpeta `apps/cfe-notas-obra` del repo (un nivel arriba de `notebooks/`) y luego el archivo **app.yaml**.
2. Deja su contenido asi, con **tus** valores de los pasos 2 y 4:

```yaml
command:
  - python
  - app_v3.py

env:
  - name: LAKEBASE_ENDPOINT
    value: "projects/<tu-proyecto>/branches/production/endpoints/primary"
  - name: PGHOST
    value: "<host del paso 4>"
  - name: PGPORT
    value: "5432"
  - name: PGDATABASE
    value: "databricks_postgres"
  - name: PGUSER
    value: "<CLIENT_ID de la app del paso 2>"
```

3. Los cambios se guardan solos. Revisa que no queden `<` `>` en el archivo.

## Paso 7 - Desplegar y probar la app

**Desplegar**
1. Ve a **Compute** -> **Apps** -> tu app.
2. Clic en **Deploy** (Desplegar).
3. En *Source code path* selecciona la carpeta del repo `.../workshopcfe/apps/cfe-notas-obra`.
4. Confirma con **Deploy**. En la pestana **Deployments** el estado debe pasar a **Succeeded** (1-2 min).
5. Si algo falla, revisa la pestana **Logs**.

**Probar**
1. Abre la **URL** de la app (aparece en la parte superior de su pagina).
2. Selecciona una obra, escribe una nota y clic en **Registrar Nota**.
3. En la tabla de notas recientes cambia el estado: **Abierta -> En Proceso -> Resuelta**.

**Confirmar en Lakebase** (SQL Editor del paso 5):

```sql
SELECT id, id_obra, tipo_nota, prioridad, estado, usuario, fecha_registro
FROM notas_obra ORDER BY id DESC LIMIT 10;

SELECT accion, detalle, fecha FROM bitacora_app ORDER BY id DESC LIMIT 10;
```

Las tablas `notas_obra` y `bitacora_app` las creo la app en su primer uso.

## Paso 8 - Sincronizar Lakebase -> Delta (Jobs)

Las notas viven en Lakebase (transaccional). Para analizarlas en dashboards, Genie o el modelo de riesgo, el notebook `sync_lakebase_to_delta` las copia a tablas Delta `notas_obra_sync`, `bitacora_app_sync` y `obras_catalogo_sync`.

**Ajustar el notebook de sync**
1. Abre `sync_lakebase_to_delta` en `apps/cfe-notas-obra`.
2. Cambia `LAKEBASE_ENDPOINT` y `PGHOST` por tus valores del paso 4. Si tu catalogo/esquema no es `dipidev.default`, ajusta tambien `CATALOG` y `SCHEMA`.

**Crear el job**
1. Menu lateral -> **Jobs & Pipelines** -> **Create** -> **Job**.
2. Nombre: `CFE - Sync Lakebase a Delta (<tus-iniciales>)`.
3. Tarea: tipo **Notebook**, ruta `.../apps/cfe-notas-obra/sync_lakebase_to_delta`, compute **Serverless**.
4. En el panel derecho, **Add trigger** -> **Scheduled** -> cada **1 hora**.
5. Clic en **Run now** y espera a que termine en verde.

**Verificar en Catalog Explorer**
* Abre **Catalog** -> `dipidev` -> `default` y busca `notas_obra_sync`. En **Sample Data** deben verse las notas que registraste.

## Troubleshooting

| Sintoma en la app | Causa probable | Solucion |
| --- | --- | --- |
| Lista de obras con `password authentication failed` | La app no tiene rol en Lakebase o `PGUSER` es incorrecto | Revisar paso 3 y el Client ID en `app.yaml` |
| `permission denied for table obras_catalogo` | Falta el `GRANT SELECT` | Repetir paso 5 |
| `permission denied for schema public` | Falta `GRANT USAGE, CREATE` | Repetir paso 5 |
| `relation "obras_catalogo" does not exist` | No se cargo el catalogo o es otro proyecto | Repetir paso 4 con el proyecto correcto |
| Deploy en estado *Failed* | Error en `app.yaml` o en el codigo | Pestana **Logs** de la app |
| La primera carga tarda unos segundos | El compute de Lakebase escalo a cero | Es normal; despierta solo |

## Limpieza (al terminar el workshop)

* **Detener la app**: pagina de la app -> **Stop** (deja de consumir compute).
* **Pausar el job** de sync: Jobs & Pipelines -> tu job -> pausar el trigger.
* **Borrar el proyecto de Lakebase** (opcional): elimina todas las notas de forma permanente.